# HireMind – Experiment Notebook

## Milestone 1: Environment Verification

In [1]:
import sys
print(sys.executable)

/home/pushkar/Desktop/hiremind/backend/.venv/bin/python


In [2]:
import fitz
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import spacy
import re

## Milestone 2: Resume PDF Parse

In [ ]:
file_path = "sample_resume.pdf"
doc = fitz.open(file_path)
print("Total pages in resume:", len(doc))

resume_text = ""

for page in doc:
    resume_text += page.get_text()

doc.close()

print("\nResume Preview:\n")
print(resume_text[:100])

Total pages in resume: 1

Resume Preview:

Sourabh Bajaj
Email : mail@website.com
http://www.sourabhbajaj.com
Mobile : +1-123-456-7890
Educatio


## Milestone 3: Semantic Resume-JD Match

In [ ]:
job_description = """
We are looking for a Backend Software Engineer
with experience in Python, Docker, Kubernetes,
AWS, REST APIs and Microservices.
"""

model = SentenceTransformer("all-MiniLM-L6-v2")

resume_embedding = model.encode(resume_text, convert_to_tensor=False)
jd_embedding = model.encode(job_description, convert_to_tensor=False)

similarity = cosine_similarity([resume_embedding], [jd_embedding])[0][0]

semantic_match_score = round(similarity * 100, 2)

print("Semantic Match Score:", semantic_match_score, "%")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic Match Score: 30.2 %


## Milestone 4: Semantic Skill Match

In [ ]:
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer("all-MiniLM-L6-v2")

skill_library = [
    "python", 
    "java", 
    "javascript", 
    "typescript", 
    "fastapi", 
    "django", 
    "flask", 
    "spring boot", 
    "docker", 
    "kubernetes", 
    "aws", 
    "azure", 
    "gcp", 
    "postgresql", 
    "mongodb", 
    "redis", 
    "graphql", 
    "rest api", 
    "microservices", 
    "machine learning", 
    "deep learning"]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
doc = nlp(resume_text)
resume_phrases = set()

for chunk in doc.noun_chunks:
    phrase = chunk.text.strip()
    if len(phrase) > 2:
        resume_phrases.add(
            phrase.lower()
        )

print("Resume Phrases Found:")
for phrase in list(resume_phrases)[:20]:
    print("-", phrase)

phrase_embeddings = model.encode(list(resume_phrases))
skill_embeddings = model.encode(skill_library)

skill_results = []

Resume Phrases Found:
- ◦quantdesk
- scala
- a set
- dataduct integration
- time
- setup
- instructors
- machine learning
- runners
- learner activity
- assignments
- • recommendation system
- numerical computation
- amazon redshift
- aws
- project trends
- electrical
- a thin rest layer
- the coursera platform
- electronics


In [7]:
for idx, skill in enumerate(skill_library):
    best_similarity = 0
    best_phrase = None

    for phrase_idx, phrase in enumerate(resume_phrases):
        similarity = cosine_similarity([skill_embeddings[idx]], [phrase_embeddings[phrase_idx]])[0][0]
        if similarity > best_similarity:
            best_similarity = similarity
            best_phrase = phrase

    skill_results.append({
        "skill": skill,
        "evidence": best_phrase,
        "similarity": round(best_similarity, 3)
    })

print("\nSKILL RESULTS:\n")
for skill in sorted(skill_results, key=lambda x: x["similarity"], reverse=True):
    print(
        f"{skill['skill']} "
        f"-> {skill['evidence']} "
        f"({skill['similarity']})"
    )


SKILL RESULTS:

python -> python (1.0)
javascript -> javascript (1.0)
docker -> docker (1.0)
aws -> aws (1.0)
machine learning -> machine learning (1.0)
deep learning -> deep learning models (0.8790000081062317)
java -> java
technologies (0.8069999814033508)
gcp -> gce (0.5979999899864197)
microservices -> service (0.5519999861717224)
postgresql -> sql (0.5090000033378601)
graphql -> ﬂow graphs (0.5080000162124634)
django -> python (0.4959999918937683)
flask -> python (0.47699999809265137)
redis -> redshift (0.46399998664855957)
azure -> aws (0.4399999976158142)
rest api -> a thin rest layer (0.4309999942779541)
typescript -> react (0.40799999237060547)
mongodb -> ◦data collection (0.38499999046325684)
fastapi -> coursera (0.33399999141693115)
kubernetes -> kafka (0.3269999921321869)
spring boot -> core service (0.2800000011920929)


## Milestone 5: Skill Gap Intelligence + Scoring

In [8]:
MATCH_THRESHOLD = 0.75
IMPROVEMENT_THRESHOLD = 0.50

matched_skills = []
improvement_areas = []
critical_gaps = []

total_similarity = 0

for result in skill_results:
    similarity = result["similarity"]
    total_similarity += similarity
    if similarity >= MATCH_THRESHOLD:
        matched_skills.append(result)
    elif similarity >= IMPROVEMENT_THRESHOLD:
        improvement_areas.append(result)
    else:
        critical_gaps.append(result)

if len(skill_results) > 0:
    match_score = round((total_similarity/len(skill_results)) * 100, 2)
else:
    match_score = 0

analysis_result = {
    "match_score": match_score,
    "matched_skills": matched_skills,
    "improvement_areas": improvement_areas,
    "critical_gaps": critical_gaps
}

In [9]:
print(f"Semantic Match Score: {match_score}%")
print("\nMATCHED SKILLS:\n")
for skill in matched_skills:
    print(
        f"{skill['skill']} "
        f"({skill['similarity']}) "
        f"← {skill['evidence']}"
    )

print("\nIMPROVEMENT AREAS:\n")
for skill in improvement_areas:
    print(
        f"{skill['skill']} "
        f"({skill['similarity']}) "
        f"← {skill['evidence']}"
    )

print("\nCRITICAL GAPS:\n")
for skill in critical_gaps:
    print(
        f"{skill['skill']} "
        f"({skill['similarity']})"
    )

Semantic Match Score: 61.400001525878906%

MATCHED SKILLS:

python (1.0) ← python
java (0.8069999814033508) ← java
technologies
javascript (1.0) ← javascript
docker (1.0) ← docker
aws (1.0) ← aws
machine learning (1.0) ← machine learning
deep learning (0.8790000081062317) ← deep learning models

IMPROVEMENT AREAS:

gcp (0.5979999899864197) ← gce
postgresql (0.5090000033378601) ← sql
graphql (0.5080000162124634) ← ﬂow graphs
microservices (0.5519999861717224) ← service

CRITICAL GAPS:

typescript (0.40799999237060547)
fastapi (0.33399999141693115)
django (0.4959999918937683)
flask (0.47699999809265137)
spring boot (0.2800000011920929)
kubernetes (0.3269999921321869)
azure (0.4399999976158142)
mongodb (0.38499999046325684)
redis (0.46399998664855957)
rest api (0.4309999942779541)


## Milestone 6: Resume Bullet Optimizer

In [ ]:
ACTION_VERBS = {
    "built",
    "developed",
    "implemented",
    "engineered",
    "created",
    "designed",
    "integrated",
    "optimized",
    "automated",
    "deployed",
    "led",
    "architected"
}

IMPACT_WORDS = {
    "reduced",
    "improved",
    "increased",
    "optimized",
    "enhanced",
    "accelerated",
    "saved",
    "boosted",
    "scaled"
}

def contains_metric(text):
    metric_patterns = [
        r"\d+%",          # 35%
        r"\d+\+",         # 12+
        r"\d+x",          # 5x
        r"\d+\s*ms",      # 300ms
        r"\d+\s*sec",     # 5 sec
        r"\d+\s*users",   # 1000 users
        r"\d+\s*api",     # 12 api
        r"\d+"            # fallback
    ]

    for pattern in metric_patterns:
        if re.search(pattern, text.lower()):
            return True
    return False

In [11]:
def contains_action_verb(text):
    words = text.lower().split()
    return any(
        word in ACTION_VERBS
        for word in words
    )

def contains_impact(text):
    words = text.lower().split()
    return any(
        word in IMPACT_WORDS
        for word in words
    )

def analyze_bullet(bullet):
    has_action = contains_action_verb(bullet)
    has_metric = contains_metric(bullet)
    has_impact = contains_impact(bullet)

    strength_score = sum([has_action, has_metric, has_impact])

    needs_optimization = (strength_score < 2)

    return {
        "bullet": bullet,
        "has_action": has_action,
        "has_metric": has_metric,
        "has_impact": has_impact,
        "strength_score": strength_score,
        "needs_optimization": needs_optimization
    }

In [12]:
resume_bullets = [
    "Worked on backend APIs.",
    "Developed 12+ REST APIs using FastAPI reducing response latency by 35%.",
    "Built dashboard for users.",
    "Implemented JWT authentication with rate limiting reducing unauthorized access attempts."
]
results = []

for bullet in resume_bullets:
    results.append(analyze_bullet(bullet))

for result in results:
    print(result)

{'bullet': 'Worked on backend APIs.', 'has_action': False, 'has_metric': False, 'has_impact': False, 'strength_score': 0, 'needs_optimization': True}
{'bullet': 'Developed 12+ REST APIs using FastAPI reducing response latency by 35%.', 'has_action': True, 'has_metric': True, 'has_impact': False, 'strength_score': 2, 'needs_optimization': False}
{'bullet': 'Built dashboard for users.', 'has_action': True, 'has_metric': False, 'has_impact': False, 'strength_score': 1, 'needs_optimization': True}
{'bullet': 'Implemented JWT authentication with rate limiting reducing unauthorized access attempts.', 'has_action': True, 'has_metric': False, 'has_impact': False, 'strength_score': 1, 'needs_optimization': True}


In [13]:
# from groq import Groq

# client = Groq(
#     api_key="YOUR_GROQ_API_KEY"
# )

def optimize_bullet(bullet):
    prompt = f"""
You are an expert technical resume writer.

Rewrite the resume bullet below.

Rules:
1. Return ONLY the rewritten bullet.
2. No explanations.
3. No markdown.
4. Keep it under 30 words.
5. Add realistic impact if possible.
6. Keep technologies intact.
7. Professional ATS-friendly tone.

Resume Bullet:
{bullet}
"""

    # response = client.chat.completions.create(
    #     model="llama-3.3-70b-versatile",
    #     messages=[{"role": "user", "content": prompt}],
    #     temperature=0.2
    # )
    # return response.choices[0].message.content.strip()

In [14]:
optimized_results = []

for result in results:
    bullet = result["bullet"]
    if result["needs_optimization"]:
        optimized_bullet = optimize_bullet(bullet)
        optimized_results.append({
            "original": bullet,
            "optimized": optimized_bullet,
            "optimized_by_llm": True
        })
    else:
        optimized_results.append({
            "original": bullet,
            "optimized": bullet,
            "optimized_by_llm": False
        })

for item in optimized_results:
    print("\n" + "=" * 80)
    print("ORIGINAL :", item["original"])
    print("OPTIMIZED:", item["optimized"])
    print("LLM_USED :", item["optimized_by_llm"])


ORIGINAL : Worked on backend APIs.
OPTIMIZED: None
LLM_USED : True

ORIGINAL : Developed 12+ REST APIs using FastAPI reducing response latency by 35%.
OPTIMIZED: Developed 12+ REST APIs using FastAPI reducing response latency by 35%.
LLM_USED : False

ORIGINAL : Built dashboard for users.
OPTIMIZED: None
LLM_USED : True

ORIGINAL : Implemented JWT authentication with rate limiting reducing unauthorized access attempts.
OPTIMIZED: None
LLM_USED : True
